# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>


# <font color="#003660">Session 7: Fine-Tuning LLMs</font>

# <font color="#003660">Notebook 2: Incorporating Human Preferences in LLMs</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... know the basics of Reinforcement Learning from Human Feedback (RLHF) and Direct Preference Optimization (DPO) <br>
        ... are able to DP optimize an LLM using the Transformers, PEFT and TRL libraries from huggingface.
    </font>
</div>
</p>

The following content is heavily inspired by the following excellent sources:


* [TRL Llama 2 Research Projects](https://github.com/huggingface/trl/tree/main/examples/research_projects/stack_llama_2/scripts)

* Diverse papers referred to in the markdown texts

## DPO
In the last notebook, we alread talked about DPO.

*Direct Preference Optimization (DPO)* ([Rafailov et al., 2023](https://doi.org/10.48550/arXiv.2305.18290)) shown in the right of the above below.

![RLHF](https://huyenchip.com/assets/pics/rlhf/6-sft-rlhf.png)

In comparison to the reward provided by the reward model as a ranking for five model answers in RLHF, DPO uses the loss divergence between the one chosen and one rejected answer.

![DPO](https://miro.medium.com/v2/resize:fit:720/format:webp/1*AqKOT0pxzi5kOgiobb-Fvg.png)


(Image source: [João Lages Blog](https://medium.com/@joaolages/direct-preference-optimization-dpo-622fc1f18707))

## Installing and Setup

In [ ]:
!pip install -U datasets transformers accelerate bitsandbytes peft trl wandb

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import AutoPeftModelForCausalLM, LoraConfig
from trl import DPOConfig, DPOTrainer

MODEL_NAME = "skaltenp/Llama-3.2-3B-Instruct-Thinking"

So now we need a thinking dataset for our DPO.

In [ ]:
dataset = load_dataset("minchyeom/Thinker-XML-DPO", split="train")

In [ ]:
import json
print(json.dumps(dataset[0], indent=4))

Looks good but our model was trained with `<think>`tags. Furthermore, the [DPOTrainer](https://huggingface.co/docs/trl/main/dataset_formats#preference) needs specific dataset format. Let's adapt the dataset.

In [ ]:
def prep_example(example):
    example["prompt"] = [{"role": "user", "content": example["prompt"]}]
    example["chosen"] = [{"role": "assistant", "content": example["chosen"].replace("<im_reasoning>", "<think>").replace("</im_reasoning>", "</think>")}]
    example["rejected"] = [{"role": "assistant", "content": example["rejected"].replace("<im_reasoning>", "<think>").replace("</im_reasoning>", "</think>")}]
    return example
dataset_prep = dataset.map(prep_example)

In [ ]:
print(json.dumps(dataset_prep[0], indent=4))

So now let's load the model (quantized):

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# loading the quantization config, 4-bit mode currently one of the smallest
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # Load the model in 4-bit mode
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoPeftModelForCausalLM.from_pretrained(
    MODEL_NAME, # location of saved SFT model
    low_cpu_mem_usage=True,
    dtype=torch.float16,
    quantization_config=bnb_config,
    is_trainable=True,
)

model_ref = AutoPeftModelForCausalLM.from_pretrained(
    MODEL_NAME, # location of saved SFT model
    low_cpu_mem_usage=True,
    dtype=torch.float16,
    quantization_config=bnb_config,
)

Here again, we setup LoRA and define the training parameters.

In [ ]:
training_args = DPOConfig(
    output_dir="Llama3.2-3B-Thinking-DPO",
    report_to="none",
    max_steps=200,
    push_to_hub=True,
)

Let's train it:

In [ ]:
dpo_trainer = DPOTrainer(
    model,
    model_ref,
    args=training_args,
    train_dataset=dataset_prep,
    processing_class=tokenizer,
)
dpo_trainer.train()

Let's checkout the model.

In [ ]:
# helper function
def generate(messages, model):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=False)[0]
    return response

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_type=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    "skaltenp/Llama3.2-3B-Thinking-DPO",
    quantization_config=bnb_config,
    device_map="auto",
)
print(generate([{"role": "user", "content": "What is the meaning of life?"}], model))

Wow. Nicely thinking about the world.

# Merge and Unload

Your turn! Merge the model and upload it again.